# StyleBot — Protocolo común de la Entrega 3

**Responsable:** Krysto Alberto Adan Jauregui Zúñiga  
**Propósito:** fijar las reglas compartidas antes de entrenar Q-Learning o Dyna-Q. Este notebook no entrena modelos.

La Entrega 3 usa la CNN congelada de la entrega anterior como percepción y un almacén 8×8 como entorno de decisión. Todos los integrantes deben conservar estas categorías, acciones, recompensas, semillas y rutas.

In [1]:
SEED = 42
SEED_DEMO = 2026
GRID_SIZE = 8
SIN_CARGA = 10
N_ESTADOS = GRID_SIZE * GRID_SIZE * 11 * 2
N_ACCIONES = 6
PRENDAS_DEMO = 50
MAX_PASOS_POR_PRENDA = 60

CATEGORIAS = [
    'Camiseta/top', 'Pantalón', 'Suéter', 'Vestido', 'Abrigo',
    'Sandalia', 'Camisa', 'Tenis', 'Bolso', 'Botín'
]
ACCIONES = {0: 'arriba', 1: 'abajo', 2: 'izquierda', 3: 'derecha', 4: 'tomar', 5: 'colocar'}
RECOMPENSAS = {'correcto': 10, 'incorrecto': -8, 'lleno': -3, 'devuelta': -3, 'invalida': -2, 'paso': -1}
MODELO_CNN = 'modelos/stylebot_cnn_final.keras'

print(f'Protocolo cargado: {len(CATEGORIAS)} categorías, {len(ACCIONES)} acciones y semilla {SEED}.')

Protocolo cargado: 10 categorías, 6 acciones y semilla 42.


## Mapa común

`E` es estante, `R` recepción, `#` obstáculo y `.` pasillo. Los estantes 0 a 7 están en la fila superior; los estantes 8 y 9 están a los lados.

In [2]:
MAPA = [
    'EEEEEEEE',
    '........',
    '.##..##.',
    '........',
    'E.#..#.E',
    '........',
    '.#....#.',
    '...RR...',
]
RECEPCION = [(7, 3), (7, 4)]
ESTANTES = {**{categoria: (0, categoria) for categoria in range(8)}, 8: (4, 0), 9: (4, 7)}
OBSTACULOS = {(fila, columna) for fila, linea in enumerate(MAPA) for columna, celda in enumerate(linea) if celda == '#'}

for fila in MAPA:
    print(' '.join(fila))
print('Recepción:', RECEPCION)
print('Estantes:', len(ESTANTES))

E E E E E E E E
. . . . . . . .
. # # . . # # .
. . . . . . . .
E . # . . # . E
. . . . . . . .
. # . . . . # .
. . . R R . . .
Recepción: [(7, 3), (7, 4)]
Estantes: 10


## Codificación del estado

El estado contiene la posición del robot, la categoría percibida que transporta (0–9 o 10 sin carga) y si el estante de esa categoría está lleno. Así se obtienen `64 × 11 × 2 = 1,408` estados manejables para aprendizaje tabular.

In [3]:
def codificar_estado(fila, columna, carga, destino_lleno):
    if not (0 <= fila < 8 and 0 <= columna < 8):
        raise ValueError('La posición debe estar dentro del mapa 8×8.')
    if not 0 <= carga <= SIN_CARGA:
        raise ValueError('La carga debe ser una categoría 0–9 o 10 sin carga.')
    return ((fila * 8 + columna) * 11 + carga) * 2 + int(bool(destino_lleno))

def decodificar_estado(estado):
    if not 0 <= int(estado) < N_ESTADOS:
        raise ValueError('Estado fuera del intervalo 0–1407.')
    valor = int(estado)
    destino_lleno = bool(valor % 2)
    valor //= 2
    carga = valor % 11
    celda = valor // 11
    return celda // 8, celda % 8, carga, destino_lleno

for estado in range(N_ESTADOS):
    assert codificar_estado(*decodificar_estado(estado)) == estado

def celdas_alcanzables(origen):
    visitadas = {origen}
    pendientes = [origen]
    while pendientes:
        fila, columna = pendientes.pop(0)
        for df, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            vecina = (fila + df, columna + dc)
            if 0 <= vecina[0] < 8 and 0 <= vecina[1] < 8 and vecina not in OBSTACULOS and vecina not in visitadas:
                visitadas.add(vecina)
                pendientes.append(vecina)
    return visitadas

alcanzables = celdas_alcanzables(RECEPCION[0])
assert all(posicion in alcanzables for posicion in ESTANTES.values())
print(f'Codificación reversible comprobada en los {N_ESTADOS:,} estados.')
print('Todos los estantes son alcanzables desde recepción.')

Codificación reversible comprobada en los 1,408 estados.
Todos los estantes son alcanzables desde recepción.


## Entrenamiento y evaluación común

Se probarán al menos dos configuraciones por algoritmo. La comparación final usará los mismos episodios para Q-Learning, Dyna-Q y el agente aleatorio. En evaluación no hay exploración ni actualización de la tabla Q.

In [4]:
CONFIGURACIONES = {
    'A': {'alpha': 0.10, 'gamma': 0.95, 'epsilon': 1.00, 'epsilon_min': 0.05, 'epsilon_decay': 0.995},
    'B': {'alpha': 0.20, 'gamma': 0.99, 'epsilon': 1.00, 'epsilon_min': 0.05, 'epsilon_decay': 0.997},
}
PASOS_PLANEACION_DYNAQ = [10, 30]
SEMILLAS_EVALUACION = list(range(1000, 1050))
METRICAS = [
    'recompensa_acumulada', 'tasa_exito', 'pasos_por_prenda',
    'colocaciones_incorrectas', 'intentos_estante_lleno',
    'acciones_invalidas', 'tiempo_entrenamiento', 'tipo_de_fallo'
]
ARCHIVOS_ESPERADOS = [
    'modelos/entrega3/q_table_qlearning.npz',
    'modelos/entrega3/q_table_dynaq.npz',
    'resultados/entrega3/comparacion_agentes.csv',
    'resultados/entrega3/configuraciones_hiperparametros.csv',
    'resultados/entrega3/curva_recompensa.png',
    'resultados/entrega3/comparacion_tasa_exito.png',
    'resultados/entrega3/analisis_fallos.csv',
    'demo/entrega3/stylebot_50_prendas.gif',
]

print('Configuraciones obligatorias:', ', '.join(CONFIGURACIONES))
print('Métricas obligatorias:', len(METRICAS))
print(f'Evaluación reproducible: semillas {SEMILLAS_EVALUACION[0]} a {SEMILLAS_EVALUACION[-1]}.')

Configuraciones obligatorias: A, B
Métricas obligatorias: 8
Evaluación reproducible: semillas 1000 a 1049.


## Separación de errores

- **Error de CNN:** el estante seleccionado coincide con la categoría predicha, pero esta predicción es distinta de la etiqueta real.
- **Error de política:** la colocación es incorrecta y el estante seleccionado no coincide con la predicción.

La etiqueta real se usa para evaluar la recompensa, pero nunca se entrega al agente como parte del estado.

In [5]:
assert len(CATEGORIAS) == 10 and len(set(CATEGORIAS)) == 10
assert set(ACCIONES) == set(range(N_ACCIONES))
assert RECOMPENSAS['correcto'] > 0
assert all(RECOMPENSAS[clave] < 0 for clave in ['incorrecto', 'lleno', 'invalida', 'paso'])
assert N_ESTADOS == 1408
assert len(ESTANTES) == 10 and len(set(ESTANTES.values())) == 10
assert PRENDAS_DEMO == 50
assert MODELO_CNN.endswith('stylebot_cnn_final.keras')
print('VALIDACIÓN FINAL: protocolo común de la Entrega 3 correcto.')

VALIDACIÓN FINAL: protocolo común de la Entrega 3 correcto.


## Siguiente paso de cada responsable

- **Luis:** implementar y entrenar Q-Learning y Dyna-Q respetando este MDP.
- **Juan José:** cargar la CNN final congelada y entregar la categoría percibida al entorno.
- **Ángel Uriel:** evaluar los agentes, producir gráficas y separar los tipos de fallo.
- **Jacqueline:** integrar `E3_StyleBot.ipynb`, informe y demostración de 50 prendas.
- **Krysto:** revisar compatibilidad, resultados, reproducibilidad y entrega final.